In [1]:
from marubatsu import TwoBitBoard3x3

def calc_legal_moves(self):
    board = 0b010101010101010101 & (self.board | (self.board >> 1))
    board = 0b010101010101010101 - board
    if self.crossturn:
        board <<= 1
    legal_moves = []
    # board が 0 になるまで繰り返し処理を行う
    while board:
        # 次の合法手である LOB を計算する
        move = board & (-board)
        legal_moves.append(move)
        # board の RSB を 0 にして move のデータを削除する
        board -= move

    return legal_moves

TwoBitBoard3x3.calc_legal_moves = calc_legal_moves

In [2]:
from marubatsu import Marubatsu

mb = Marubatsu(boardclass = TwoBitBoard3x3)
print(mb)
for move in mb.calc_legal_moves():
    print(mb.board.move_to_xy(move))
print()
mb.cmove(0, 0)
print(mb)
for move in mb.calc_legal_moves():
    print(mb.board.move_to_xy(move))

Turn o
...
...
...

(0, 0)
(0, 1)
(0, 2)
(1, 0)
(1, 1)
(1, 2)
(2, 0)
(2, 1)
(2, 2)

Turn x
O..
...
...

(0, 1)
(0, 2)
(1, 0)
(1, 1)
(1, 2)
(2, 0)
(2, 1)
(2, 2)


In [3]:
from marubatsu import BitBoard3x3

mb1 = Marubatsu(boardclass=BitBoard3x3)
mb2 = Marubatsu(boardclass=TwoBitBoard3x3)
print(mb1)
%timeit mb1.calc_legal_moves()
%timeit mb2.calc_legal_moves()
for x, y in [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 1), (2, 0)]:
    mb1.cmove(x, y)
    mb2.cmove(x, y)
    print(mb1)
    %timeit mb1.calc_legal_moves()
    %timeit mb2.calc_legal_moves()

Turn o
...
...
...

955 ns ± 17.5 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
1.06 μs ± 4.85 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
Turn x
O..
...
...

858 ns ± 4.01 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
1.05 μs ± 6.37 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
Turn o
o..
X..
...

781 ns ± 5.11 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
880 ns ± 14.1 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
Turn x
o..
x..
O..

692 ns ± 4.27 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
882 ns ± 4.34 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
Turn o
oX.
x..
o..

624 ns ± 5.46 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
736 ns ± 6.73 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
Turn x
ox.
xO.
o..

524 ns ± 2.01 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
720 ns ± 7.34 ns per loop (mean

In [4]:
from ai import ai_match, ai2
import random

random.seed(0)
ai_match(ai=[ai2, ai2], mbparams={"boardclass": TwoBitBoard3x3}, match_num=50000)

ai2 VS ai2


100%|██████████| 50000/50000 [00:02<00:00, 21945.68it/s]

count     win    lose    draw
o       29454   14352    6194
x       14208   29592    6200
total   43662   43944   12394

ratio     win    lose    draw
o       58.9%   28.7%   12.4%
x       28.4%   59.2%   12.4%
total   43.7%   43.9%   12.4%



[('count',
  [{'win': 29454, 'lose': 14352, 'draw': 6194},
   {'win': 14208, 'lose': 29592, 'draw': 6200},
   {'win': 43662, 'lose': 43944, 'draw': 12394}],
  '7d'),
 ('ratio',
  [{'win': 0.58908, 'lose': 0.28704, 'draw': 0.12388},
   {'win': 0.28416, 'lose': 0.59184, 'draw': 0.124},
   {'win': 0.43662, 'lose': 0.43944, 'draw': 0.12394}],
  '7.1%')]

In [5]:
TwoBitBoard3x3.ptable = [i.bit_count() for i in range(1 << 18)]
TwoBitBoard3x3.masklist = [
    [0, 0b000000000000010101, 0b000000000000101010], # 0 列のビットマスク
    [0, 0b000000010101000000, 0b000000101010000000], # 1 列のビットマスク
    [0, 0b010101000000000000, 0b101010000000000000], # 2 列のビットマスク
    [0, 0b000001000001000001, 0b000010000010000010], # 0 行のビットマスク
    [0, 0b000100000100000100, 0b001000001000001000], # 1 行のビットマスク
    [0, 0b010000010000010000, 0b100000100000100000], # 2 行のビットマスク
    [0, 0b010000000100000001, 0b100000001000000010], # 対角線 1 のビットマスク
    [0, 0b000001000100010000, 0b000010001000100000], # 対角線 2 のビットマスク
]  

In [6]:
from collections import defaultdict

def count_markpats(self, turn, last_turn):
    markpats = defaultdict(int)
    for mask in self.masklist:
        turncount = self.ptable[self.board & mask[turn]]
        lastturncount = self.ptable[self.board & mask[last_turn]]
        emptycount = self.BOARD_SIZE - turncount - lastturncount
        markpats[(lastturncount, turncount, emptycount)] += 1

    return markpats  

TwoBitBoard3x3.count_markpats = count_markpats

In [7]:
mb1 = Marubatsu(boardclass=BitBoard3x3)
mb2 = Marubatsu(boardclass=TwoBitBoard3x3)
mb1.cmove(0, 0)
mb1.cmove(1, 0)
mb1.cmove(0, 1)
mb2.cmove(0, 0)
mb2.cmove(1, 0)
mb2.cmove(0, 1)
print(mb1)
print(mb1.count_markpats())
print(mb2.count_markpats())
%timeit mb1.count_markpats()
%timeit mb2.count_markpats()

Turn x
ox.
O..
...

defaultdict(<class 'int'>, {(2, 0, 1): 1, (0, 1, 2): 1, (0, 0, 3): 3, (1, 1, 1): 1, (1, 0, 2): 2})
defaultdict(<class 'int'>, {(2, 0, 1): 1, (0, 1, 2): 1, (0, 0, 3): 3, (1, 1, 1): 1, (1, 0, 2): 2})
2.92 μs ± 11.2 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
2.84 μs ± 10.7 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [8]:
from ai import ai14s

ai_match(ai=[ai14s, ai2], mbparams={"boardclass": TwoBitBoard3x3}, match_num=50000)

ai14s VS ai2


100%|██████████| 50000/50000 [00:16<00:00, 3117.85it/s]

count     win    lose    draw
o       49446       0     554
x       44043       0    5957
total   93489       0    6511

ratio     win    lose    draw
o       98.9%    0.0%    1.1%
x       88.1%    0.0%   11.9%
total   93.5%    0.0%    6.5%



[('count',
  [{'win': 49446, 'lose': 0, 'draw': 554},
   {'win': 44043, 'lose': 0, 'draw': 5957},
   {'win': 93489, 'lose': 0, 'draw': 6511}],
  '7d'),
 ('ratio',
  [{'win': 0.98892, 'lose': 0.0, 'draw': 0.01108},
   {'win': 0.88086, 'lose': 0.0, 'draw': 0.11914},
   {'win': 0.93489, 'lose': 0.0, 'draw': 0.06511}],
  '7.1%')]

In [9]:
def board_to_hashable(self):
    return self.board

TwoBitBoard3x3.board_to_hashable = board_to_hashable

In [10]:
%timeit mb1.board_to_hashable()
%timeit mb2.board_to_hashable()

195 ns ± 2.67 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)
90.3 ns ± 0.142 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


In [11]:
def board_to_str(self):
    txt = ""
    for x in range(self.BOARD_SIZE):
        for y in range(self.BOARD_SIZE):
            txt += self.MARK_TABLE[self.getmark(x, y)]
    return txt

TwoBitBoard3x3.board_to_str = board_to_str

In [12]:
print(mb1.board_to_str())
print(mb2.board_to_str())

oo.x.....
oo.x.....


In [13]:
def calc_same_hashables(self, move=None):
    if move is None:
        hashables = set()
    else:
        hashables = {}
    if move is not None:
        x, y = self.move_to_xy(move)
    board = self.board
    for _ in range(4):
        # 左右の反転処理
        c = (board ^ (board >> 12)) & 0b111111
        board = c ^ (c << 12) ^ board
        if move is None:
            hashables.add(board)
        else:
            x = self.BOARD_SIZE - x - 1
            hashables[board] = self.xy_to_move(x, y)
            
        # 転置処理
        c = (board ^ (board >> 4)) & 0b110000001100
        board = c ^ (c << 4) ^ board 
        c = (board ^ (board >> 8)) & 0b110000
        board = c ^ (c << 8) ^ board 
        if move is None:
            hashables.add(board)
        else:
            x, y = y, x
            hashables[board] = self.xy_to_move(x, y)
    return hashables   

TwoBitBoard3x3.calc_same_hashables = calc_same_hashables

In [14]:
mb = Marubatsu(boardclass=TwoBitBoard3x3)
mb.cmove(1, 1)
mb.cmove(0, 0)
mb.cmove(1, 0)
move = mb.board.xy_to_move(1, 0)
hashables = mb.board.calc_same_hashables(move)
for hashable, move in hashables.items():
    mb.board.board = hashable
    print(mb.board.move_to_xy(move))
    print(mb)
    print()

(1, 0)
Turn x
.Ox
.o.
...


(0, 1)
Turn x
...
oo.
x..


(2, 1)
Turn x
...
.oo
..x


(1, 2)
Turn x
...
.o.
.ox


(1, 2)
Turn x
...
.o.
xo.


(2, 1)
Turn x
..x
.oo
...


(0, 1)
Turn x
x..
oo.
...


(1, 0)
Turn x
xO.
.o.
...




In [15]:
%timeit mb1.board.calc_same_hashables()
%timeit mb2.board.calc_same_hashables()

4.26 μs ± 15.2 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
2.65 μs ± 13.8 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [16]:
from util import benchmark

benchmark(mbparams={"boardclass":TwoBitBoard3x3})

ai2 VS ai2


100%|██████████| 50000/50000 [00:02<00:00, 21774.56it/s]


count     win    lose    draw
o       29454   14352    6194
x       14208   29592    6200
total   43662   43944   12394

ratio     win    lose    draw
o       58.9%   28.7%   12.4%
x       28.4%   59.2%   12.4%
total   43.7%   43.9%   12.4%

ai14s VS ai2


100%|██████████| 50000/50000 [00:15<00:00, 3197.78it/s]


count     win    lose    draw
o       49446       0     554
x       44043       0    5957
total   93489       0    6511

ratio     win    lose    draw
o       98.9%    0.0%    1.1%
x       88.1%    0.0%   11.9%
total   93.5%    0.0%    6.5%

ai_abs_dls
  6.4 ms ±   0.3 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
def __init__(self, boardclass=None, board_size=3, *args, **kwargs):
    if boardclass is None:
            boardclass = BitBoard3x3
    # ゲーム盤のデータ構造を定義するクラス
    self.boardclass = boardclass
    # ゲーム盤の縦横のサイズ
    self.BOARD_SIZE = board_size
    # boardclass のパラメータ
    self.args = args
    self.kwargs = kwargs
    self.board = self.boardclass(self.BOARD_SIZE, *self.args, **self.kwargs)
    self.EMPTY = self.board.EMPTY
    self.CIRCLE = self.board.CIRCLE
    self.CROSS = self.board.CROSS
    # 〇×ゲーム盤を再起動するメソッドを呼び出す
    self.restart()
    
Marubatsu.__init__ = __init__